# 05 -- Phase 4: Diagnostics

`cassa-diagnose` inspects each stage and writes a multi-page PDF, per-stage
PNGs, and a machine-readable `metrics.json`. This is where you *read* the
error budget you carried from raw pixels to calibrated magnitudes.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG  --  EDIT THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================
import os

# 1) Shared, read-only Astrometry.net index directory (used by Phase 2):
os.environ["CASSA_ASTROMETRY_INDEX"] = os.path.abspath("../../astrometry_data")

# 2) The provided workshop dataset + a writable work directory:
RAW_DIR   = "../raw"     # provided raw frames, one level up from notebooks/
WORK_DIR  = "../work"    # writable output dir, one level up from notebooks/

RAW_DIR   = os.path.abspath(os.path.expandvars(RAW_DIR))
WORK_DIR  = os.path.abspath(os.path.expandvars(WORK_DIR))

# These are relative to the working directory, which Jupyter sets to this
# notebook's folder. Fail loudly here rather than confusingly further down.
assert os.path.isdir(RAW_DIR), (
    f"RAW_DIR not found: {RAW_DIR}\nRun this notebook from the notebooks/ "
    f"directory, or set RAW_DIR/WORK_DIR to absolute paths above."
)

# Each phase writes into its own directory under WORK_DIR.
PHASE1_DIR = os.path.join(WORK_DIR, "phase1")   # calibrated frames
PHASE2_DIR = os.path.join(WORK_DIR, "phase2")   # master stacks + WCS
PHASE3_DIR = os.path.join(WORK_DIR, "phase3")   # flux-calibrated + catalogs
PHASE4_DIR = os.path.join(WORK_DIR, "phase4")   # diagnostics report
os.makedirs(WORK_DIR, exist_ok=True)
print("RAW_DIR    =", RAW_DIR)
print("PHASE1_DIR =", PHASE1_DIR)
print("PHASE2_DIR =", PHASE2_DIR)

In [ ]:
import glob, os
assert os.path.isdir(PHASE2_DIR), "No phase2 directory -- run notebook 03 (Phase 2) first."
print("Phase 2 (masters)  :", PHASE2_DIR)
print("Phase 3 (catalogs) :", PHASE3_DIR)

## Run diagnostics across all stages

In [ ]:
from cassa_photometry.phase4_diagnostics import run as diag_run
diag_run(run_dir=PHASE2_DIR, raw=RAW_DIR, outdir=PHASE4_DIR)
print('Report dir:', PHASE4_DIR)

## Read the metrics summary

In [ ]:
import json, os, glob
mjson = os.path.join(outdir, 'metrics.json')
if os.path.exists(mjson):
    metrics = json.load(open(mjson))
    print(json.dumps(metrics, indent=2)[:2000])
else:
    print('metrics.json not found; check:', glob.glob(os.path.join(outdir, '*')))

The full **`diagnostics_report.pdf`** in that directory has one page per
stage (raw -> calibrated -> master -> photometry). Open it to review the PSF,
DQ accounting, depth boost, and the zero-point scatter.

That completes the end-to-end run: raw frames -> calibrated catalog.